# Lilly — Mapillary Harvest + Crop (GPU)

Harvests street photos from Mapillary for Bosnian/Croatian/Serbian cities,
then crops text regions with EasyOCR on GPU.

Output: crops-mapillary.zip (PNG crops + labels.tsv)

Token: attach a Kaggle secret named MAPILLARY_TOKEN or attach
a dataset lilly-mapillary-token with token.txt inside.

In [ ]:
# 1. Sanity checks
import os, subprocess, sys, urllib.request, urllib.error, shutil, json, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), "No GPU. Enable GPU in Session options."
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.get_device_name(0))

TEE = Path("/kaggle/working/stdout.txt")

def run(*cmd, quiet=False):
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "
")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
        for out in child.stdout:
            if not quiet: print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code: raise subprocess.CalledProcessError(code, cmd)


In [ ]:
# 2. Mapillary token — from secret or attached dataset
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("MAPILLARY_TOKEN")
    print("token from Kaggle secret")
except Exception:
    # fallback: dataset with token.txt
    candidates = list(Path("/kaggle/input").rglob("token.txt"))
    if not candidates:
        raise SystemExit("No MAPILLARY_TOKEN secret and no token.txt in input")
    TOKEN = candidates[0].read_text(encoding="utf-8").strip()
    print("token from", candidates[0])
os.environ["MAPILLARY_TOKEN"] = TOKEN
assert TOKEN.startswith("MLY|"), f"token looks wrong: {TOKEN[:8]}..."
print("token ok")


In [ ]:
# 3. Clone repo to scratch
SCRATCH = Path("/kaggle/temp")
SCRATCH.mkdir(exist_ok=True)
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
os.chdir(CLONE)
print("working in", os.getcwd())


In [ ]:
# 4. Install
run(sys.executable, "-m", "pip", "install", "-q",
    "easyocr", "opencv-python-headless", "pillow")


In [ ]:
# 5. Download Bosnia + Croatia + Serbia OSM extracts for signage points
# Bosnia already in repo (points.tsv). HR+RS need the script.
run("python3", "data/scripts/fetch_osm_extract.py")
run("python3", "data/scripts/fetch_hr_rs_extract.py")
npoints = sum(1 for _ in open("data/signs/points.tsv")) - 1
print(f"{npoints:,} signage points in cache")
assert npoints > 30000, f"too few points: {npoints}"


In [ ]:
# 6. Harvest photos — all cities, no text filter, 5000 per city
# GPU not needed here, but network is fast on Kaggle
CITIES = [
    "sarajevo", "mostar", "tuzla", "zenica", "banjaluka",
    "zagreb", "split", "dubrovnik", "rijeka", "osijek",
    "beograd", "novi_sad", "nis",
]
for city in CITIES:
    run("python3", "data/scripts/harvest_mapillary.py",
        "--city", city, "--limit", "5000", "--no-filter")

photo_dir = Path("data/ocr/real-photos/mapillary")
photos = list(photo_dir.glob("mly_*.jpg"))
print(f"
Harvested {len(photos):,} photos total")
assert len(photos) >= 5000, f"harvest too thin: {len(photos)}"


In [ ]:
# 7. Crop text regions from every photo — GPU EasyOCR
import easyocr
from PIL import Image

print("Loading EasyOCR on GPU...", flush=True)
reader = easyocr.Reader(["bs", "en"], gpu=True)
print("Reader ready", flush=True)

CROPS_DIR = Path("/kaggle/working/crops-mapillary")
CROPS_DIR.mkdir(exist_ok=True)
labels_rows = []
MIN_CONF = 0.4

for i, photo in enumerate(sorted(photos)):
    try:
        image = Image.open(photo).convert("RGB")
        regions = reader.readtext(str(photo), detail=1)
    except Exception as exc:
        print(f"  SKIP {photo.name}: {exc}", flush=True)
        continue
    for j, (box, text, conf) in enumerate(regions):
        text = text.strip()
        if not text or conf < MIN_CONF:
            continue
        xs = [int(p[0]) for p in box]
        ys = [int(p[1]) for p in box]
        x0, y0 = max(min(xs), 0), max(min(ys), 0)
        x1, y1 = max(xs), max(ys)
        crop = image.crop((x0, y0, x1, y1))
        if crop.width < 8 or crop.height < 8:
            continue
        name = f"{photo.stem}_{j:03d}.png"
        crop.save(CROPS_DIR / name)
        labels_rows.append(f"{name}	{text}	{conf:.2f}")
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(photos)} photos | {len(labels_rows):,} crops", flush=True)

labels_file = CROPS_DIR / "labels.tsv"
with open(labels_file, "w", encoding="utf-8") as f:
    f.write("file	text	confidence
")
    f.write("
".join(labels_rows) + "
")

print(f"
{len(labels_rows):,} crops from {len(photos):,} photos")
print(f"labels -> {labels_file}")
assert len(labels_rows) >= 1000, f"too few crops: {len(labels_rows)}"


In [ ]:
# 8. Zip and report
run("zip", "-qr", "/kaggle/working/crops-mapillary.zip",
    str(CROPS_DIR.relative_to("/kaggle/working")))
zipsize = Path("/kaggle/working/crops-mapillary.zip").stat().st_size
print(f"crops-mapillary.zip: {zipsize / 1048576:.0f} MB")
print(f"
Done: {len(labels_rows):,} crops ready for OCR training")

with TEE.open("a") as sink:
    sink.write(f"CROPS: {len(labels_rows)}
")
    sink.write(f"PHOTOS: {len(photos)}
")
